<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/SectorStrengthScore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta
!pip install scipy==1.16.2

  Using cached ta-0.11.0.tar.gz (25 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=f5556bfb145d2a9c016adaf65f23ded40fa9fe5b25788807926282b60e290262
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 23.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


Sector Score=w1(RS)+ w2(Momentum)+ w3(Breadth)+ w4(Trend) + w5(Efficiency)

Example weights (start here):
RS → 30%
Momentum → 25%
Breadth → 20%
Trend slope → 15%
Volatility efficiency → 10%

In [105]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import linregress
# ensure reproducibility
import random
random.seed(42)
print("Libraries Installed!")

Libraries Installed!


# Short Term Sector Rotation Strategy

In [106]:
class STSectorRotationRanker:
    def __init__(self, price_data: pd.DataFrame, sector_etfs: list, benchmark: str = 'SPY'):
        """
        price_data: DataFrame of adjusted closes with Date index (ascending) and columns for tickers (including benchmark).
        sector_etfs: list of sector ETF tickers to rank (strings).
        benchmark: ticker string present in price_data used as market benchmark (default 'SPY').

        """
        self.price_data = price_data.copy().sort_index()
        self.sector_etfs = sector_etfs
        self.benchmark = benchmark

        if benchmark not in self.price_data.columns:
            raise ValueError(f"Benchmark {benchmark} not found in price_data columns.")

    def _relative_series(self, ticker):
        """Return ratio series ticker / benchmark (RS series)."""
        return self.price_data[ticker] / self.price_data[self.benchmark]

    def compute_basic_rs(self, short_window=21, mid_window=63, long_window=126):
        """Compute RS1M, RS3M and RS6M as relative % change vs benchmark over short/mid windows."""
        results = []
        for t in self.sector_etfs:
            if t not in self.price_data.columns:
                results.append({'ticker': t, 'RS1M': np.nan, 'RS3M': np.nan,'RS6M':np.nan})
                continue
            rel = self._relative_series(t)
            #rs1m = (rel.iloc[-1] / rel.shift(short_window).iloc[-1] - 1) if len(rel) > short_window else np.nan
            #rs3m = (rel.iloc[-1] / rel.shift(mid_window).iloc[-1] - 1) if len(rel) > mid_window else np.nan
            rs1m = rel.pct_change(short_window).iloc[-1] if len(rel) > short_window else np.nan
            rs3m = rel.pct_change(mid_window).iloc[-1] if len(rel) > mid_window else np.nan
            rs6m = rel.pct_change(long_window).iloc[-1] if len(rel) > long_window else np.nan
            results.append({'ticker': t, 'RS1M': rs1m, 'RS3M': rs3m,"RS6M":rs6m})
        return pd.DataFrame(results).set_index('ticker')

    def compute_rs_slope(self, window=21):
        """Compute linear regression slope of RS series over `window` bars (last window)."""
        slopes = {}
        x = np.arange(window)
        for t in self.sector_etfs:
            if t not in self.price_data.columns:
                slopes[t] = np.nan
                continue
            rel = self._relative_series(t).dropna()
            if len(rel) < window:
                slopes[t] = np.nan
                continue
            y = rel.values[-window:]
            slope, intercept, r, p, se = linregress(x, y)
            slopes[t] = slope
        return pd.Series(slopes, name='RS_Slope')

    def compute_rolling_rs_acceleration(self, slope_window=21, accel_window=5):
        """
        Compute acceleration = change in rolling slope over accel_window steps.
        slope_window: how many bars to fit each rolling slope
        accel_window: how many slope-steps to look back for acceleration
        """
        acc = {}
        for t in self.sector_etfs:
            if t not in self.price_data.columns:
                acc[t] = np.nan
                continue
            rel = self._relative_series(t).dropna()
            if len(rel) < slope_window + accel_window:
                acc[t] = np.nan
                continue
            slopes = []
            for i in range(len(rel) - slope_window + 1):
                y = rel.values[i:i+slope_window]
                x = np.arange(slope_window)
                slope = linregress(x, y)[0]
                slopes.append(slope)
            slopes = np.array(slopes)
            acc_val = slopes[-1] - slopes[-1-accel_window] if len(slopes) > accel_window else np.nan
            acc[t] = acc_val
        return pd.Series(acc, name='RS_Acceleration')

    def merge_with_breadth(self):
      """
      Merge RS factors with breadth metrics
      """
      rs = self.compute_basic_rs()
      slope = self.compute_rs_slope()
      accel = self.compute_rolling_rs_acceleration()
      merged_df = rs.join(slope).join(accel)
      return merged_df

    def compute_leader_score(self, rs1w=0.5, rs3w=0.3, rs6w=0.2,slopew=0, accelw=0):
        """
        rs1w=0.4, rs3w=0.4, slopew=0.15, accelw=0.05
        Produce a DataFrame with RS1M, RS3M, RS6M, RS_Slope, RS_Acceleration and Leader_Score.
        Leader_Score computed as weighted z-scores of metrics.
        """
        rs = self.compute_basic_rs()
        slope = self.compute_rs_slope()
        accel = self.compute_rolling_rs_acceleration()

        df = rs.join(slope).join(accel)

        def zscore_col(s):
            # Use standard zscore; safe-guard against zero std
            return (s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) != 0 else 1)

        z_rs1   = zscore_col(df['RS1M'])
        z_rs3   = zscore_col(df['RS3M'])
        z_rs6   = zscore_col(df['RS6M'])
        z_slope = zscore_col(df['RS_Slope'])
        z_accel = zscore_col(df['RS_Acceleration'])

        score = rs1w*z_rs1 + rs3w*z_rs3 + rs6w*z_rs6 + slopew*z_slope + accelw*z_accel
        df['Leader_Score'] = score
        return df[['RS1M','RS3M', 'RS6M','RS_Slope','RS_Acceleration','Leader_Score']]

    def final_ranking(self):
      """
       Return final ranking DataFrame. If breadth_df provided it will be joined.
       Adds a new column 'Rank' where highest Leader_Score gets rank 1.
      """
      # Compute the leader score
      merged = self.compute_leader_score()
      # Sort by Leader_Score descending
      merged = merged.sort_values('Leader_Score', ascending=False)

      # Add a new ranking column
      merged['Rank'] = merged['Leader_Score'].rank(method='first', ascending=False).astype(int)


      return merged

In [107]:
start_date = "2023-10-01"
end_date   = "2026-03-31"
week_end_date   = "2026-03-31"


tickers = ["XLF", "XLK", "XLV", "XLE", "XLY", "XLP", "XLI", "XLU", "XLRE","XLB","XLC","GBTC","GLD","MAGS","SPY"]
sectors = ["XLF", "XLK", "XLV", "XLE", "XLY", "XLP", "XLI", "XLU", "XLRE","XLB","XLC","GBTC","GLD","MAGS"]

if __name__ == "__main__":



  data = yf.download( tickers, start=start_date, end=end_date, interval="1d",auto_adjust=True)['Close']
  price_data = data.ffill().dropna()
  # Instantiate and run ----------
  ranker = STSectorRotationRanker(price_data=price_data, sector_etfs=sectors, benchmark='SPY')
  df_ranking = ranker.final_ranking()

  #print(breadth_df)
def label_rs1m(rs1m):
    if rs1m >= 0.06:  return "BULLISH ≥6%"
    if rs1m >= 0.02:  return "Positive +2–6%"
    if rs1m >= -0.02: return "Neutral ±2%"
    if rs1m >= -0.05: return "BEARISH –2 to –5%"
    return "COLLAPSING <–5%"

def label_rs3m(rs3m):
    if rs3m >= 0.08:  return "LEADING ≥8%"
    if rs3m >= 0.00:  return "Recovering 0–8%"
    if rs3m >= -0.05: return "Lagging –5 to 0%"
    return "DEAD <–5%"

def label_rs_slope(slope):
    if slope >= 0.0045:      return "ROCKET / Parabolic"
    if slope >= 0.0030:      return "Very Strong Uptrend"
    if slope >= 0.0020:      return "Strong Uptrend"
    if slope >= 0.0012:      return "Moderate Uptrend"
    if slope >= 0.0006:      return "Mild Uptrend"
    if slope >= 0.0000:      return "Flat / Barely Positive"
    if slope >= -0.0010:     return "Flat / Losing Ground"
    return "Downtrend — Avoid"
def interpret_acceleration(acc):
      if acc >= 0.00030: return "ROCKET / Exploding"
      if acc >= 0.00020: return "Very strong acceleration"
      if acc >= 0.00012: return "Strong acceleration"
      if acc >= 0.00008: return "Positive acceleration"
      if acc >= 0.00000: return "Neutral/slightly positive"
      return "Deceleration / avoid"

def regime(row):
    if row['Slope_Label'].startswith('ROCKET') or row['Accel_Description'].startswith('ROCKET'):
        return "MONSTER INCOMING"
    if 'Strong' in row['Slope_Label'] or 'Very Strong' in row['Accel_Description']:
        return "Clear Leadership"
    if row['RS_Slope'] > 0.0015 and row['RS_Acceleration'] > 0.00015:
        return "Strong & Accelerating"
    if row['RS_Slope'] > 0:
        return "Improving"
    return "Wait / Avoid"



df_ranking['RS1M_Label'] = df_ranking['RS1M'].apply(label_rs1m)
df_ranking['RS3M_Label'] = df_ranking['RS3M'].apply(label_rs3m)
df_ranking['Slope_Label'] = df_ranking['RS_Slope'].apply(label_rs_slope)
df_ranking['Accel_Description'] = df_ranking['RS_Acceleration'].apply(interpret_acceleration)
df_ranking['Regime'] = df_ranking.apply(regime, axis=1)

def quick_verdict(row):
    bad_short = row['RS1M'] < -0.02                     # short-term momentum against us
    bad_long  = row['RS3M'] < -0.08                     # still in deep bear market
    monster   = (row['RS_Slope'] >= 0.0035) or (row['RS_Acceleration'] >= 0.00025)

    if bad_short and not monster:
        return "AVOID — short-term momentum too negative"
    if bad_long and row['Rank'] <= 2:
        return "CAUTION — still deeply underwater"
    if row['RS1M'] >= 0.04 and row['RS_Slope'] >= 0.002:
        return "SCREAMING BUY"
    return "Good"

df_ranking['Quick_Verdict'] = df_ranking.apply(quick_verdict, axis=1)


df_ranking

[*********************100%***********************]  15 of 15 completed


,RS1M,RS3M,RS6M,RS_Slope,RS_Acceleration,Leader_Score,Rank,RS1M_Label,RS3M_Label,Slope_Label,Accel_Description,Regime,Quick_Verdict
ticker,,,,,,,,,,,,,
XLE,0.207284,0.537024,0.423945,0.000814,0.000256,2.759033,1,BULLISH ≥6%,LEADING ≥8%,Mild Uptrend,Very strong acceleration,Improving,Good
XLU,0.048753,0.177471,0.117653,0.000111,0.000040,0.568693,2,Positive +2–6%,LEADING ≥8%,Flat / Barely Positive,Neutral/slightly positive,Improving,Good
XLB,-0.000611,0.164899,0.161864,0.000003,0.000280,0.222480,3,Neutral ±2%,LEADING ≥8%,Flat / Barely Positive,Very strong acceleration,Improving,Good
XLP,-0.009712,0.146583,0.108643,-0.000064,0.000223,0.061009,4,Neutral ±2%,LEADING ≥8%,Flat / Losing Ground,Very strong acceleration,Wait / Avoid,Good
XLRE,-0.000438,0.088271,0.017501,-0.000065,-0.000029,-0.076297,5,Neutral ±2%,LEADING ≥8%,Flat / Losing Ground,Deceleration / avoid,Wait / Avoid,Good
XLF,0.023136,-0.047982,-0.056940,0.000065,0.000098,-0.227532,6,Positive +2–6%,Lagging –5 to 0%,Flat / Barely Positive,Positive acceleration,Improving,Good
GBTC,0.095437,-0.175988,-0.370962,0.000172,-0.000380,-0.264071,7,BULLISH ≥6%,DEAD <–5%,Flat / Barely Positive,Deceleration / avoid,Improving,Good
XLV,-0.024167,0.008093,0.114512,-0.000267,0.000214,-0.291678,8,BEARISH –2 to –5%,Recovering 0–8%,Flat / Losing Ground,Very strong acceleration,Wait / Avoid,AVOID — short-term momentum too negative
XLI,-0.040249,0.088231,0.075482,-0.000378,0.000179,-0.312268,9,BEARISH –2 to –5%,LEADING ≥8%,Flat / Losing Ground,Strong acceleration,Wait / Avoid,AVOID — short-term momentum too negative


# ⚙️ CONFIGURATION

In [3]:
WEIGHTS = {
    "RS": 0.30,
    "Momentum": 0.25,
    "Breadth": 0.20,
    "Trend": 0.15,
    "Efficiency": 0.10
}

LOOKBACKS = {
    "wtd": 5,
    "mtd": 21,
    "3m": 63
}


# 📥 DATA FETCHING

In [20]:
def fetch_prices(tickers, period="6mo"):
    data = yf.download(tickers, period=period, auto_adjust=True)["Close"]
    return data.dropna(axis=1, how="all")


# 📊 METRIC FUNCTIONS

In [4]:
def compute_returns(df, period):
    return df.pct_change(period).iloc[-1]

def compute_momentum(df):
    wtd = compute_returns(df, LOOKBACKS["wtd"])
    mtd = compute_returns(df, LOOKBACKS["mtd"])
    m3 = compute_returns(df, LOOKBACKS["3m"])
    return 0.2 * wtd + 0.4 * mtd + 0.4 * m3

def compute_relative_strength(df, benchmark):
    sector_returns = compute_returns(df, LOOKBACKS["3m"])
    benchmark_return = compute_returns(benchmark.to_frame(), LOOKBACKS["3m"]).iloc[0]
    return sector_returns / benchmark_return

def compute_trend_slope(df, window=20):
    ma = df.rolling(window).mean()
    return (ma.iloc[-1] - ma.iloc[-window]) / window

def compute_efficiency(df, window=20):
    returns = df.pct_change()
    vol = returns.rolling(window).std()
    total_return = df.pct_change(window).iloc[-1]
    return abs(total_return) / vol.iloc[-1]


# 📊 BREADTH (FROM YOUR INPUT)

In [12]:
def compute_breadth_from_percentages(breadth_dict):
    """
    Input format:
    {
        "XLK": {"ma20": 0.72, "ma50": 0.65},
        ...
    }
    """
    scores = {}

    for sector, values in breadth_dict.items():
        ma20 = values.get("ma20", np.nan)
        ma50 = values.get("ma50", np.nan)

        # Weighted breadth score
        scores[sector] = 0.6 * ma20 + 0.4 * ma50

    return pd.Series(scores)

# 🔄 NORMALIZATION

In [13]:
def normalize(series):
    if series.max() == series.min():
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

# 🧠 SCORING ENGINE

In [15]:
def compute_scores(df_prices, benchmark_prices, breadth_series=None):

    momentum = compute_momentum(df_prices)
    rs = compute_relative_strength(df_prices, benchmark_prices)
    trend = compute_trend_slope(df_prices)
    efficiency = compute_efficiency(df_prices)

    df = pd.DataFrame({
        "RS": rs,
        "Momentum": momentum,
        "Trend": trend,
        "Efficiency": efficiency
    })

    # Add breadth if available
    if breadth_series is not None:
        df["Breadth"] = breadth_series

    # Normalize all metrics
    for col in df.columns:
        df[col] = normalize(df[col])

    # Select only available metrics
    active_weights = {k: v for k, v in WEIGHTS.items() if k in df.columns}

    # Re-normalize weights
    total_weight = sum(active_weights.values())
    active_weights = {k: v / total_weight for k, v in active_weights.items()}

    # Compute final score
    df["Score"] = 0
    for metric, weight in active_weights.items():
        df["Score"] += df[metric] * weight

    df["Rank"] = df["Score"].rank(ascending=False)

    return df.sort_values("Score", ascending=False)

# 🌍 MARKET CONFIGURATION

In [98]:
MARKETS = {
    "US": {
        "sectors": ["XLK", "XLC", "XLY", "XLRE", "XLF", "XLP","XLI", "XLU", "XLE", "XLB","XLV" ],
        "benchmark": "^GSPC"
    }

}

# ▶️ RUN ANALYSIS

In [99]:
def run_market_analysis(market_name, config, breadth_input=None):
    print(f"\n===== {market_name} =====")

    # Fetch prices
    sector_prices = fetch_prices(config["sectors"])
    benchmark_prices = fetch_prices([config["benchmark"]]).squeeze()

    # Handle optional breadth
    breadth_series = None
    if breadth_input is not None and market_name in breadth_input:
        breadth_series = compute_breadth_from_percentages(
            breadth_input[market_name]
        )

    # Compute scores
    results = compute_scores(sector_prices, benchmark_prices, breadth_series)

    #print(results)
    return results

# 🚀 EXECUTION

In [100]:
# Example Breadth Input (OPTIONAL)
breadth_input = {
    "US": {
        "XLY": {"ma20": 0.73, "ma50": 0.51},
        "XLP": {"ma20": 0.49, "ma50": 0.29},
        "XLE": {"ma20": 0.41, "ma50": 0.59},
        "XLF": {"ma20": 0.70, "ma50": 0.66},
        "XLV": {"ma20": 0.28, "ma50": 0.24},
        "XLI" :{"ma20": 0.65, "ma50": 0.52},
        "XLK": {"ma20": 0.77, "ma50": 0.74},
        "XLB": {"ma20": 0.46, "ma50": 0.46},
        "XLRE": {"ma20": 0.94, "ma50": 0.68},
        "XLC": {"ma20": 0.48, "ma50": 0.52},
        "XLU": {"ma20": 0.29, "ma50": 0.39},
    }
}

all_results = {}

for market, config in MARKETS.items():
    try:
        results = run_market_analysis(market, config, breadth_input)
        all_results[market] = results

        # Save results
        results.to_csv(f"{market}_sector_ranking.csv")

    except Exception as e:
        print(f"Error in {market}: {e}")

def combine_results(all_results):
    frames = []

    for market, df in all_results.items():
        temp = df.copy()
        temp["Market"] = market
        temp["Sector"] = temp.index
        frames.append(temp)

    combined_df = pd.concat(frames, axis=0)
    combined_df = combined_df.reset_index(drop=True)
    desired_order = [
        "Market",
        "Sector",
        "Global_Rank",
        "RS",
        "Momentum",
        "Trend",
        "Efficiency",
        "Breadth",
        "Score",
        "Rank"
    ]
        # keep only columns that actually exist
    cols = [c for c in desired_order if c in combined_df.columns]

    return combined_df[cols]

df = combine_results(all_results)
df
#df.to_csv("combined_results.csv")


===== US =====


[*********************100%***********************]  11 of 11 completed
[*********************100%***********************]  1 of 1 completed


,Market,Sector,RS,Momentum,Trend,Efficiency,Breadth,Score,Rank
0,US,XLK,0.762537,1.000000,1.000000,1.000000,0.863636,0.901488,1.0
1,US,XLRE,0.615362,0.642259,0.355482,0.706085,1.000000,0.669104,2.0
2,US,XLE,1.000000,0.565078,0.269054,0.300108,0.381119,0.587862,3.0
3,US,XLI,0.547847,0.504618,0.432155,0.296661,0.583916,0.501781,4.0
4,US,XLB,0.508790,0.504612,0.449480,0.402050,0.342657,0.454949,5.0
5,US,XLU,0.709296,0.550325,0.281723,0.077690,0.115385,0.423474,6.0
6,US,XLY,0.190562,0.346665,0.448873,0.394274,0.660839,0.382761,7.0
7,US,XLF,0.223350,0.287116,0.414753,0.258676,0.734266,0.373718,8.0
8,US,XLC,0.293414,0.302966,0.280574,0.386553,0.405594,0.325626,9.0
9,US,XLP,0.369112,0.358127,0.118505,0.141502,0.255245,0.283240,10.0


## Other markets

In [35]:
# =========================================
# DEFAULT NORMALIZATION
# =========================================

def normalize(series):
    if series.max() == series.min():
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / (series.max() - series.min())


# =========================================
# UNIVERSAL SCORING ENGINE
# =========================================

def compute_sector_score(df, weights, normalize_rs=True):
    """
    Universal scoring engine for TSX, ASX, NGX, etc.

    df: DataFrame with required features
    weights: dict of feature weights
    """

    df = df.copy()

    # --- Normalize RS if present ---
    if normalize_rs:
        for col in df.columns:
            if "RS" in col:
                df[col] = normalize(df[col])

    # --- Compute Score ---
    df["Score"] = 0

    for col, w in weights.items():
        if col in df.columns:
            df["Score"] += df[col] * w

    df["Rank"] = df["Score"].rank(ascending=False)

    return df.sort_values("Score", ascending=False)

🌍 MARKET CONFIGS (THIS IS THE REAL POWER)

In [82]:
# 🇨🇦 TSX CONFIG
TSX_WEIGHTS = {
    "RS_1M_angle": 0.15,  # early rotation signal
    "RS_3M_angle": 0.25,  # structuralleadership
    "RS_3M_above_zero": 0.2,
    "above_10W": 0.10,    # short trend alignment
    "above_30W": 0.15,    # regime support
    "price_angle_10W": 0.05,  # timing confirmation
    "price_angle_30W": 0.10   # regime filter (slow anchor)
}

# 🇦🇺 ASX CONFIG (slightly more momentum-sensitive)
ASX_WEIGHTS = {
    "RS_1M_angle": 0.20,  # early rotation signal
    "RS_3M_angle": 0.25,  # structuralleadership
    "RS_3M_above_zero": 0.15,
    "above_10W": 0.10,    # short trend alignment
    "above_30W": 0.10,    # regime support
    "price_angle_10W": 0.10,  # timing confirmation
    "price_angle_30W": 0.10   # regime filter (slow anchor)

}

# 🇳🇬 NGX CONFIG (your binary structure model)Slightly more weight on structure (because NGX is trend-driven)
NGX_WEIGHTS = {
    "RS_1M_angle": 0.25,  # early rotation signal
    "RS_3M_angle": 0.20,  # structuralleadership
    "RS_3M_above_zero": 0.10,
    "above_10W": 0.10,    # short trend alignment
    "above_30W": 0.10,    # regime support
    "price_angle_10W": 0.15,  # timing confirmation
    "price_angle_30W": 0.10   # regime filter (slow anchor)
}

SP500_WEIGHTS = {
    "RS_1M_angle": 0.15,        # still useful but not dominant
    "RS_3M_angle": 0.30,        # core leadership driver
    "RS_3M_above_zero": 0.15,   # regime confirmation (important in US)
    "above_10W": 0.10,          # short trend alignment
    "above_30W": 0.10,          # structural regime filter
    "price_angle_13W": 0.10,    # timing confirmation
    "price_angle_30W": 0.10     # macro trend anchor
}

def dict_to_sector_df(sector_dict):
    df = pd.DataFrame.from_dict(sector_dict, orient="index")
    df.index.name = "Sector"
    return df

## NGX Input Dictionary

In [52]:
ngx_input_dict = {
    "Consumer Goods": {
        "RS_1M_angle": 53.71, # MA length 5
        "RS_3M_angle": 38.47, # MA length 13
        "RS_3M_above_zero": 0,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 9.13,   # 10W SMA TIMING
        "price_angle_30W": 12.1    # 30W SMA regime confirmation
    },

    "Industrials": {
        "RS_1M_angle": 48.62,  # MA length 5
        "RS_3M_angle": 27.91,  # MA length 13
        "RS_3M_above_zero": 1,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 33.76,   # 10W SMA TIMING
        "price_angle_30W": 19.55    # 30W SMA regime confirmation
    },

    "Oil and Gas": {
        "RS_1M_angle": -56.45,  # MA length 5
        "RS_3M_angle": -39.77,  # MA length 13
        "RS_3M_above_zero": 1,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 42.83,   # 10W SMA TIMING
        "price_angle_30W": 25.01    # 30W SMA regime confirmation
    },

    "Banking": {
        "RS_1M_angle": 20.72,  # MA length 5
        "RS_3M_angle": 24.42,  # MA length 13
        "RS_3M_above_zero": 1,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 42.24,   # 10W SMA TIMING
        "price_angle_30W": 16.84    # 30W SMA regime confirmation
    },

     "Insurance": {
        "RS_1M_angle": -4.33,  # MA length 5
        "RS_3M_angle": -2.17,  # MA length 13
        "RS_3M_above_zero": 0,
        "above_10W": 0,
        "above_30W": 0,
        "price_angle_10W": -13.47,   # 10W SMA TIMING
        "price_angle_30W": -3.61    # 30W SMA regime confirmation
    }
}

# NGX ranking
df_ngx_input = dict_to_sector_df(ngx_input_dict)
#df_ngx_input
df_ngx = compute_sector_score(df_ngx_input, NGX_WEIGHTS)
df_ngx

,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Sector,,,,,,,,,
Oil and Gas,0.000000,0.000000,1.0,1,1,42.83,25.01,9.225500,1.0
Banking,0.700527,0.820424,1.0,1,1,42.24,16.84,8.659216,2.0
Industrials,0.953794,0.865031,1.0,1,1,33.76,19.55,7.730455,3.0
Consumer Goods,1.000000,1.000000,0.0,1,1,9.13,12.10,3.229500,4.0
Insurance,0.473130,0.480573,0.0,0,0,-13.47,-3.61,-2.167103,5.0


## ASX/TSX Sector Inputs

In [110]:
market_config = {

    "SPY": {
        "benchmark": "^GSPC",
        "sectors": {
            "Financials": "XLF",
            "Energy": "XLE",
            "Materials": "XLB",
            "Industrials": "XLI",
            "Technology": "XLK",
            "Utilities": "XLU",
            "Health Care": "XLV",
            "Consumer Discretionary": "XLY",
            "Consumer Staples": "XLP",
            "Real Estate": "XLRE",
            "Biotechnology": "XBI",
            "Bitcoin": "GBTC",
            "Communication Services": "XLC",
            "Semi-Conductors": "SOXX",
            "Gold":"GLD",
            "Bonds": "TLT",
            "MAGS": "MAGS"
        }
    },


    "TSX": {
        "benchmark": "^GSPTSE",
        "sectors": {
            "Financials": "XFN.TO",
            "Energy": "XEG.TO",
            "Materials": "XMA.TO",
            "Industrials": "XGI.TO",
            "Technology": "XIT.TO",
            "Utilities": "XUT.TO",
            "Health Care": "XHC.TO",
            "Consumer Discretionary": "XCD.TO",
            "Consumer Staples": "XST.TO"
        }
    },

    "ASX": {
        "benchmark": "^AXJO",
        "sectors": {
            "Financials": "^AXFJ",
            "Materials": "^AXMJ",
            "Energy": "^AXEJ",
            "Industrials": "^AXNJ",
            "Technology": "^AXIJ",
            "Utilities": "^AXUJ",
            "Health Care": "^AXHJ",
            "Consumer Discretionary": "^AXDJ",
            "Consumer Staples": "^AXSJ",
            "Real Estate": "^AXRE",
            "Communication Services": "^AXTJ"
        }
    }
}

In [66]:
def get_data(ticker, period="3y"):
    df = yf.download(ticker, period=period, interval="1wk", auto_adjust=True)
    df = df.dropna()
    return df["Close"]


def normalize(series):
    if series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())


def angle(series):
    """
    Simple proxy for trend angle = slope of rolling regression proxy
    """
    return series.diff().rolling(4).mean()


# -----------------------------
# Core feature builder
# -----------------------------

def build_sector_features(sector_price, benchmark_price):
    aligned = pd.concat([sector_price, benchmark_price], axis=1).dropna()
    aligned.columns = ["sector", "benchmark"]

    rs = aligned["sector"] / aligned["benchmark"]

    # RS angles
    rs_1m = rs.pct_change(4)
    rs_3m = rs.pct_change(13)

    rs_1m_angle = angle(rs_1m)
    rs_3m_angle = angle(rs_3m)

    # MA structure
    ma_10w = aligned["sector"].rolling(10).mean()
    ma_30w = aligned["sector"].rolling(30).mean()

    above_10w = (aligned["sector"] > ma_10w).astype(int)
    above_30w = (aligned["sector"] > ma_30w).astype(int)

    # Price trend angles
    price_angle_10w = angle(aligned["sector"].rolling(10).mean())
    price_angle_30w = angle(ma_30w)

    # RS regime
    rs_3m_above_zero = (rs_3m > 0).astype(int)

    latest = {
        "RS_1M_angle": rs_1m_angle.iloc[-1],
        "RS_3M_angle": rs_3m_angle.iloc[-1],
        "RS_3M_above_zero": rs_3m_above_zero.iloc[-1],
        "above_10W": int(above_10w.iloc[-1]),
        "above_30W": int(above_30w.iloc[-1]),
        "price_angle_10W": price_angle_10w.iloc[-1],
        "price_angle_30W": price_angle_30w.iloc[-1],
    }

    return latest


# -----------------------------
# Market engine
# -----------------------------

def build_market_inputs(market_config):
    benchmark = get_data(market_config["benchmark"])

    output = {}

    for sector, ticker in market_config["sectors"].items():
        sector_price = get_data(ticker)

        features = build_sector_features(sector_price, benchmark)
        output[sector] = features

    return output

In [111]:
usx_data = build_market_inputs(market_config["SPY"])
df_usx = pd.DataFrame.from_dict(usx_data, orient="index")
usx_result = compute_sector_score(df_usx, SP500_WEIGHTS)

usx_result

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Semi-Conductors,1.000000,1.000000,1.0,1,1,5.254961,4.749208,1.274921,1.0
Technology,0.717204,0.927479,1.0,1,1,0.605685,0.413598,0.777184,2.0
Biotechnology,0.553661,0.681413,1.0,1,1,0.729750,1.238177,0.761291,3.0
Industrials,0.581164,0.595486,1.0,1,1,0.153925,0.661011,0.681922,4.0
Gold,0.592708,0.454663,0.0,0,1,-1.882750,3.277750,0.653080,5.0
Real Estate,0.580549,0.657364,1.0,1,1,0.142320,0.072188,0.641511,6.0
Materials,0.527096,0.484224,1.0,1,1,0.075336,0.225815,0.596913,7.0
Utilities,0.402405,0.552636,1.0,1,1,0.279990,0.156989,0.591851,8.0
MAGS,0.724135,0.888667,0.0,1,1,-0.090000,0.020004,0.577221,9.0
Consumer Discretionary,0.623754,0.730597,0.0,1,1,-0.434269,-0.121194,0.500623,10.0


In [61]:
tsx_data = build_market_inputs(market_config["TSX"])
df_tsx = pd.DataFrame.from_dict(tsx_data, orient="index")
tsx_result = compute_sector_score(df_tsx, TSX_WEIGHTS)

tsx_result

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Financials,0.782210,0.747669,1.0,1,1,0.343911,0.391911,0.810635,1.0
Industrials,0.818250,0.573454,1.0,1,1,0.093000,0.268202,0.747571,2.0
Utilities,0.495044,0.582949,1.0,1,1,0.278786,0.144987,0.698432,3.0
Energy,0.000000,0.000000,1.0,1,1,0.398253,0.260803,0.495993,4.0
Technology,0.805113,1.000000,0.0,1,0,0.234250,-0.402000,0.442279,5.0
Materials,1.000000,0.292026,0.0,0,1,0.023564,0.447544,0.418939,6.0
Consumer Discretionary,0.833658,0.657587,0.0,1,0,-0.307500,-0.069269,0.367144,7.0
Health Care,0.661917,0.479594,0.0,0,0,-0.419000,0.151382,0.213374,8.0
Consumer Staples,0.503033,0.447692,0.0,0,0,-0.075472,0.129628,0.196567,9.0


In [95]:
asx_data = build_market_inputs(market_config["ASX"])
df_asx = pd.DataFrame.from_dict(asx_data, orient="index")
#df_asx
asx_result = compute_sector_score(df_asx, ASX_WEIGHTS)

asx_result

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Energy,0.000000,0.000000,1.0,1,1,189.225000,77.602498,27.032750,1.0
Materials,0.959679,0.521304,1.0,1,1,24.867529,177.844189,20.943434,2.0
Consumer Staples,0.492433,0.569564,1.0,1,1,89.325000,22.846655,11.808043,3.0
Utilities,0.436531,0.590477,1.0,1,1,83.060010,19.085002,10.799427,4.0
Financials,0.565985,0.617050,1.0,0,1,38.122485,4.984993,4.828207,5.0
Communication Services,0.560111,0.692597,1.0,1,0,5.767502,-4.945834,0.617338,6.0
Industrials,0.649515,0.528466,0.0,0,0,-35.269995,-17.620825,-5.027063,7.0
Real Estate,0.844907,0.840398,0.0,1,0,-35.657501,-23.177502,-5.404420,8.0
Technology,1.000000,1.000000,0.0,1,0,-17.617502,-42.459166,-5.457667,9.0
Consumer Discretionary,0.642907,0.540227,0.0,0,0,-47.684998,-35.757501,-8.080612,10.0
